# Concordance 01 — scJDO

Standalone scJDO run on the bone marrow CD34+ dataset. Produces
`concordance_results/scjdo_per_cluster.npz` with the standardised
schema used by `concordance_04_compare.ipynb`.

**Environment.** `scvelo + scjdo`. **Do not** install splicejac or
dynamo in the same env — they downgrade anndata.

    pip install scvelo scjdo torch


In [1]:
import os, sys, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, torch
SEED   = 0
N_PCS  = 30
TOP_K  = 30
RESULTS_DIR = Path('concordance_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
from scjdo.tl import fit_drift
rng = np.random.default_rng(SEED); torch.manual_seed(SEED)
QUICK    = True
N_EPOCHS = 800 if QUICK else 3000
print(f'QUICK={QUICK}  N_EPOCHS={N_EPOCHS}')

QUICK=True  N_EPOCHS=800


## 1. Load + shared preprocessing

In [2]:
import scvelo as scv
import scanpy as sc
import numpy as np

adata = scv.datasets.bonemarrow()
scv.pp.filter_and_normalize(adata, min_shared_counts=20)
scv.pp.moments(adata, n_pcs=N_PCS, n_neighbors=30)
sc.tl.diffmap(adata, n_comps=15)

# Deterministic iroot — highest-CD34 cell in HSC_1
_label_col = next((c for c in ['clusters','celltype','cell_type','leiden']
                   if c in adata.obs), None)
_hsc_mask  = adata.obs[_label_col].astype(str).str.startswith('HSC').to_numpy() \
             if _label_col else np.ones(adata.n_obs, bool)
if 'CD34' in adata.var_names:
    _cd34 = adata[:, 'CD34'].X
    _cd34 = np.asarray(_cd34.todense()).ravel() if hasattr(_cd34, 'todense') \
            else np.asarray(_cd34).ravel()
else:
    _cd34 = np.zeros(adata.n_obs)
if _hsc_mask.any() and _cd34.max() > 0:
    _idx = np.flatnonzero(_hsc_mask)
    adata.uns['iroot'] = int(_idx[np.argmax(_cd34[_idx])])
else:
    adata.uns['iroot'] = int(np.argmax(_cd34)) if _cd34.max() > 0 else 0
sc.tl.dpt(adata)
adata.obs['pseudotime'] = adata.obs['dpt_pseudotime'].astype(np.float32)
pt = adata.obs['pseudotime'].to_numpy()
adata.obs['pseudotime'] = ((pt - np.nanmin(pt)) /
                            (np.nanmax(pt) - np.nanmin(pt) + 1e-9)).astype(np.float32)
adata.obs['cluster'] = adata.obs[_label_col].astype('category')
CLUSTERS   = adata.obs['cluster'].cat.categories.tolist()
GENE_NAMES = list(adata.var_names)
print(f'{adata.n_obs} cells × {adata.n_vars} genes  | '
      f'clusters: {CLUSTERS}  | iroot={adata.uns["iroot"]}')


Filtered out 7837 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.


/var/folders/8q/0m_1_8yj0r1_hxyz4br3vg4r0000gp/T/ipykernel_65943/1859992275.py:7: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata, n_pcs=N_PCS, n_neighbors=30)
/Users/terooatt/miniconda3/envs/scJDO/lib/python3.10/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(
/Users/terooatt/miniconda3/envs/scJDO/lib/python3.10/site-packages/scvelo/preprocessing/neighbors.py:233: DeprecationWarning: Automatic computation of PCA is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute PCA with Scanpy first.
  _set_pca(adata=adata, n_pcs=n_pcs, use_highly_variable=use_highly_variable)


computing neighbors
    finished (0:00:05) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities
    finished (0:00:01) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
5780 cells × 6482 genes  | clusters: ['HSC_1', 'HSC_2', 'Ery_1', 'Mono_1', 'Precursors', 'CLP', 'Mono_2', 'DCs', 'Ery_2', 'Mega']  | iroot=1722


In [3]:
import json
metadata = {
    'dataset':         'scvelo.datasets.bonemarrow (Setty 2019 CD34+)',
    'n_cells':         int(adata.n_obs),
    'n_genes':         int(adata.n_vars),
    'seed':            SEED,
    'top_k':           TOP_K,
    'n_pcs':           N_PCS,
    'cluster_order':   CLUSTERS,
    'iroot_cell_idx':  int(adata.uns['iroot']),
}
(RESULTS_DIR / 'shared_metadata.json').write_text(json.dumps(metadata, indent=2))
print('shared_metadata.json saved')


shared_metadata.json saved


## 2. scJDO — fit drift, per-cluster Jacobian at centroids

In [4]:
model = fit_drift(adata, rep='X_pca', time_key='pseudotime',
                  n_epochs=N_EPOCHS, n_archetypes=4,
                  vel_scale=2.0, seed=SEED, verbose=False)
res = adata.uns['scjdo']
print(f'fit_drift done | R²={res["r2"]:.3f} | '
      f'latent dim={adata.obsm["X_pca"].shape[1]}')

def jacobian_at(model, x, t):
    x = torch.tensor(x, dtype=torch.float32, requires_grad=True).reshape(1, -1)
    t = torch.tensor([float(t)], dtype=torch.float32)
    f = model(x, t).reshape(-1)
    J = torch.stack([torch.autograd.grad(f[i], x, retain_graph=True)[0].reshape(-1)
                     for i in range(f.shape[0])])
    return J.detach().numpy()

X_lat = adata.obsm['X_pca']
t_all = adata.obs['pseudotime'].to_numpy()
inst_per_cluster = {}
vec_latent       = {}                # cluster -> (d_lat,)
for c in CLUSTERS:
    sel = (adata.obs['cluster'] == c).to_numpy()
    if sel.sum() < 5:
        print(f'  [skip] {c}: only {int(sel.sum())} cells'); continue
    x_c = X_lat[sel].mean(0); t_c = float(np.nanmean(t_all[sel]))
    Jc  = jacobian_at(model, x_c, t_c)
    w, V = np.linalg.eig(Jc); k = int(np.argmax(w.real))
    inst_per_cluster[c] = float(w[k].real)
    vec_latent[c]       = V[:, k].real / (np.linalg.norm(V[:, k].real) + 1e-12)
print(pd.Series(inst_per_cluster).sort_values(ascending=False).round(3))

fit_drift done | R²=0.982 | latent dim=30
CLP           0.016
Mono_1        0.015
Mono_2        0.012
Ery_2         0.011
DCs           0.010
HSC_1         0.010
HSC_2         0.009
Precursors    0.009
Mega          0.009
Ery_1         0.008
dtype: float64


## 3. Project latent eigenvectors → gene space via FA loadings

In [5]:
PCs = adata.varm['PCs']                                # (n_var, n_pcs)
n_lat = next(iter(vec_latent.values())).shape[0]
PCs = PCs[:, :n_lat]

gene_vec       = np.zeros((len(CLUSTERS), len(GENE_NAMES)), dtype=np.float32)
gene_inst_rank = np.zeros_like(gene_vec)
top_genes      = {}
inst_arr       = np.full(len(CLUSTERS), np.nan, dtype=np.float32)
for i, c in enumerate(CLUSTERS):
    if c not in vec_latent: continue
    inst_arr[i] = inst_per_cluster[c]
    g  = PCs @ vec_latent[c]                           # (n_var,)
    g  = g / (np.linalg.norm(g) + 1e-12)
    gene_vec[i]       = g
    gene_inst_rank[i] = np.abs(g)
    top_idx = np.argsort(gene_inst_rank[i])[::-1][:TOP_K]
    top_genes[c] = np.array([GENE_NAMES[j] for j in top_idx])
print('top 5 genes per cluster (first 3):')
for c in CLUSTERS[:3]:
    if c in top_genes: print(f'  {c:12s}', list(top_genes[c][:5]))

top 5 genes per cluster (first 3):
  HSC_1        [np.str_('RACK1'), np.str_('IGLL1'), np.str_('CD79B'), np.str_('SPINK2'), np.str_('FOS')]
  HSC_2        [np.str_('MALAT1'), np.str_('LAPTM5'), np.str_('CD74'), np.str_('CD79B'), np.str_('IGLL1')]
  Ery_1        [np.str_('SRGN'), np.str_('VPREB1'), np.str_('PRSS57'), np.str_('HLA-DPB1'), np.str_('MALAT1')]


## 4. Save standardized .npz

In [6]:
save_path = RESULTS_DIR / 'scjdo_per_cluster.npz'
np.savez_compressed(save_path,
    method='scjdo',
    clusters=np.array(CLUSTERS),
    inst_per_cluster=inst_arr,
    gene_names=np.array(GENE_NAMES),
    gene_vec=gene_vec,
    gene_inst_rank=gene_inst_rank,
    **{f'top_genes/{c}': top_genes[c] for c in top_genes})
print(f'Saved → {save_path}  ({save_path.stat().st_size:,} bytes)')
print(f'  clusters:       {CLUSTERS}')
print(f'  inst per clu:   {inst_arr}')
print(f'  gene_vec shape: {gene_vec.shape}')

Saved → concordance_results/scjdo_per_cluster.npz  (527,761 bytes)
  clusters:       ['HSC_1', 'HSC_2', 'Ery_1', 'Mono_1', 'Precursors', 'CLP', 'Mono_2', 'DCs', 'Ery_2', 'Mega']
  inst per clu:   [0.009640045 0.009352067 0.00768027  0.014540424 0.009228914 0.015659535
 0.011740837 0.009913948 0.010730377 0.009099343]
  gene_vec shape: (10, 6482)
